# 03 — Descomposicion Frecuencia-Severidad

## Pregunta de Negocio

¿Las perdidas estan impulsadas por mas siniestros (frecuencia) o por siniestros mas costosos (severidad)? Esta descomposicion es fundamental en actuaria para:
- Tarificacion: Pure Premium = Frecuencia x Severidad
- Reaseguro: las coberturas de frecuencia vs. severidad son productos distintos
- Gestion de riesgo: las intervenciones son diferentes para cada componente

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PALETTE = {'Private Passenger Auto': '#1E3A5F', 'Commercial Auto': '#4A7FA8',
           'Workers Compensation': '#2B6B4F', 'Medical Malpractice': '#8B2E3B',
           'Other Liability': '#C4841D', 'Product Liability': '#5B4A8A'}

claims = pd.read_parquet('../data/processed/claims_synthetic.parquet')
claims['accident_year'] = claims['accident_date'].dt.year
triangles = pd.read_parquet('../data/processed/triangles.parquet')

## 1. Calculo de Frecuencia y Severidad

- **Frecuencia** = Numero de siniestros por ano de accidente
- **Severidad** = Monto incurrido promedio por siniestro
- **Pure Premium** = Frecuencia x Severidad (perdida esperada total)

In [ ]:
freq_sev = claims.groupby(['line_of_business', 'accident_year']).agg(
    frequency=('claim_id', 'count'),
    avg_severity=('incurred_amount', 'mean'),
    total_incurred=('incurred_amount', 'sum'),
    median_severity=('incurred_amount', 'median'),
).reset_index()

freq_sev['pure_premium'] = freq_sev['frequency'] * freq_sev['avg_severity']
freq_sev.head()

## 2. Tendencias de Frecuencia por LOB

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Frecuencia (siniestros/ano)', 'Severidad Promedio ($)'])

for lob, color in PALETTE.items():
    lob_data = freq_sev[freq_sev['line_of_business'] == lob]
    fig.add_trace(go.Scatter(x=lob_data['accident_year'], y=lob_data['frequency'],
                             name=lob, line=dict(color=color), legendgroup=lob), row=1, col=1)
    fig.add_trace(go.Scatter(x=lob_data['accident_year'], y=lob_data['avg_severity'],
                             name=lob, line=dict(color=color, dash='dot'), legendgroup=lob,
                             showlegend=False), row=1, col=2)

fig.update_layout(template='plotly_white', height=400, title='Tendencias de Frecuencia y Severidad',
                  font_family='Georgia')
fig.show()

## 3. Descomposicion del Pure Premium

El grafico de barras apiladas muestra como se compone la prima pura (perdida esperada) para cada LOB.

In [ ]:
# Pure premium by LOB (aggregated across years)
pp_summary = claims.groupby('line_of_business').agg(
    total_claims=('claim_id', 'count'),
    avg_severity=('incurred_amount', 'mean'),
    total_incurred=('incurred_amount', 'sum'),
).reset_index()
pp_summary['avg_frequency_per_year'] = pp_summary['total_claims'] / 10  # 10 accident years
pp_summary['pure_premium_annual'] = pp_summary['avg_frequency_per_year'] * pp_summary['avg_severity']

fig = go.Figure()
fig.add_trace(go.Bar(
    x=pp_summary['line_of_business'], y=pp_summary['avg_frequency_per_year'],
    name='Frecuencia (siniestros/ano)', marker_color='#1E3A5F',
    yaxis='y',
))
fig.add_trace(go.Scatter(
    x=pp_summary['line_of_business'], y=pp_summary['avg_severity'],
    name='Severidad Promedio ($)', marker_color='#C4841D',
    yaxis='y2', mode='markers+lines', marker_size=10,
))

fig.update_layout(
    template='plotly_white', height=400, font_family='Georgia',
    title='Frecuencia vs Severidad por LOB',
    yaxis=dict(title='Frecuencia (siniestros/ano)'),
    yaxis2=dict(title='Severidad Promedio ($)', overlaying='y', side='right'),
)
fig.show()

## 4. Year-over-Year Growth Analysis

In [ ]:
# YoY growth rates
freq_sev = freq_sev.sort_values(['line_of_business', 'accident_year'])
freq_sev['freq_yoy'] = freq_sev.groupby('line_of_business')['frequency'].pct_change() * 100
freq_sev['sev_yoy'] = freq_sev.groupby('line_of_business')['avg_severity'].pct_change() * 100
freq_sev['pp_yoy'] = freq_sev.groupby('line_of_business')['pure_premium'].pct_change() * 100

# Flag the driver
freq_sev['driver'] = np.where(
    freq_sev['freq_yoy'].abs() > freq_sev['sev_yoy'].abs(),
    'Frecuencia', 'Severidad'
)

# Summary
driver_counts = freq_sev.dropna(subset=['driver']).groupby(['line_of_business', 'driver']).size().unstack(fill_value=0)
driver_counts

## So What?

- **Auto personal**: alta frecuencia, severidad relativamente baja y estable. Las perdidas son predecibles — la ley de los grandes numeros aplica bien.
- **Malpraxis medica**: baja frecuencia pero severidad extrema y volatil. Un solo siniestro puede representar millones — requiere reaseguro de exceso de perdida.
- **Workers' comp**: frecuencia moderada con severidad en tendencia ascendente — sugiere inflacion en costos medicos y laborales.
- Para tarificacion, las lineas impulsadas por frecuencia necesitan datos granulares de exposicion, mientras que las impulsadas por severidad requieren modelos de colas pesadas (Pareto, GPD).